In [2]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
import matplotlib as mp
# import gaia_tools as gt
import scipy
from scipy.ndimage import gaussian_filter
import astropy.units as u
from astropy.coordinates import SkyCoord
import math
import h5py
# import healpy as hp
import pykdgrav3_utils
from pykdgrav3_utils import units
un = units.units(1, 600., verbose=True)
import healpy as hp
from healpy.newvisufunc import projview, newprojplot
from matplotlib.projections.geo import GeoAxes

sys.path.append('/home/dnurme/linux_env/Thesis/My_thesis/Modules/')
from mock_wake import generate_mock_wake
from rotation_funcs import rotate, angle_finder, rz, ry, rx
from misc import plot_OD_gaussian

dMsolUnit = 1.000000e+00
dKpcUnit = 6.000000e+02
dGasConst =  1150890.1952769116
dErgPerGmUnit =  71.68174956254887
dGmPerCcUnit =  3.1333829769061664e-40
dSecUnit =  2.1867420491060357e+23
dKmPerSecUnit =  8.466507518602276e-05
dComovingGmPerCcUnit =  3.1333829769061664e-40


In [3]:
file = '/home/dnurme/linux_env/Data/dm_sim.00001.0'

def load_snap_file(path, part_type='PartType1', is_print = False):

    snap_file = h5py.File(path, 'r')
    part_data = snap_file[part_type]

    if(is_print):
        print(f'Loading snapshot: {path.split("/")[-1]}')
        print(f'Selected species: {part_type}')
        print(f'Snap file keys: {snap_file.keys()}')
        print(f'Part type keys: {part_data.keys()}')

    return part_data

In [4]:
## Find LMC velocity when it is at 70 kpc distance. Also find its current position.

orbitfile = '/home/dnurme/linux_env/Data/trajlmc.txt'
orbit_full = pd.read_csv(orbitfile, delimiter = ' ')
orbit = orbit_full.loc[np.where(orbit_full['time'] < 0.01)]
d_orbit = np.sqrt(orbit['x']**2 + orbit['y']**2 + orbit['z']**2)
loc70 = np.isclose(d_orbit, 70.0, 0.01)
# 70 - d_orbit[loc70]
v70 = orbit['Vx'][loc70].values[0], orbit['Vy'][loc70].values[0], orbit['Vz'][loc70].values[0]
LMC_70 = np.array([orbit['x'][loc70].values[0], orbit['y'][loc70].values[0], orbit['z'][loc70].values[0]])

LMC_today = np.isclose(orbit['time'], 0.0, 0.001)
LMC_loc_today = np.array([orbit['x'][LMC_today].values[0], orbit['y'][LMC_today].values[0], orbit['z'][LMC_today].values[0]])

In [5]:
snap_stars = load_snap_file(file, part_type='PartType4', is_print=True)
star_coord = snap_stars['Coordinates'][:]*un.dKpcUnit

Loading snapshot: dm_sim.00001.0
Selected species: PartType4
Snap file keys: <KeysViewHDF5 ['Cosmology', 'Header', 'Parameters', 'PartType1', 'PartType4', 'Units']>
Part type keys: <KeysViewHDF5 ['Coordinates', 'GroupID', 'Masses', 'ParticleIDs', 'Potential', 'Softening', 'StellarFormationTime', 'Velocities']>


In [6]:
x = star_coord[:,0]
y = star_coord[:,1]

In [7]:
## Rotate simulation

z_angle, y_angle = angle_finder(v70)

R = rz(np.radians(z_angle)) @ ry(np.radians(y_angle))
star_coord_rot = R @ star_coord.T
star_coord_rot = star_coord_rot.T

Rz = -98.38192060960003, Ry = -5.918706233387963


In [8]:
del snap_stars
del star_coord

In [9]:
## Find perpendicular and parallel distances

a = (LMC_loc_today - LMC_70)
v_n = v70 / np.linalg.norm(v70)

d_par = np.dot(a, v_n)*v_n
d_perp = a - d_par

In [10]:
star_coord_rot += LMC_loc_today 
star_coord_rot -= d_perp


In [11]:
dist_mask = np.sqrt(star_coord_rot[:,0]**2 + star_coord_rot[:,1]**2 + star_coord_rot[:,2]**2)
masked_LMC = star_coord_rot[np.where((dist_mask > 70) & (dist_mask < 100))]

In [ ]:
LMC_sky = SkyCoord(masked_LMC[:,0], 
                   masked_LMC[:,1], 
                   masked_LMC[:,2], 
                   frame = 'galactocentric', 
                   unit='kpc', 
                   representation_type='cartesian')

LMC_orbit = SkyCoord(orbit['x'], 
                     orbit['y'], 
                     orbit['z'], 
                     frame = 'galactocentric', 
                     unit='kpc', 
                     representation_type='cartesian')

LMC_location_today = SkyCoord(LMC_loc_today[0], 
                         LMC_loc_today[1], 
                         LMC_loc_today[2], 
                         frame = 'galactocentric', 
                         unit='kpc', 
                         representation_type='cartesian')

In [ ]:
# # Healpy osa
# nside = 64

# gal_l = LMC_sky.galactic.l.wrap_at(180 * u.deg)
# gal_b = LMC_sky.galactic.b

# lon = gal_l.radian
# lat = gal_b.radian

# theta = 0.5*np.pi - lat                # latitude -> colatitude
# phi   = lon % (2*np.pi)                # wrap to [0, 2π)

# pix = hp.ang2pix(nside, theta, phi)
# m = np.bincount(pix, minlength=hp.nside2npix(nside)).astype(float)

# # Overdensity osa
# pop = m > 0
# mu  = m[pop].mean() if np.any(pop) else 1.0

# delta = np.full_like(m, hp.UNSEEN, dtype=float)
# delta[pop] = (m[pop] / mu) - 1.0

# # Smoothing funktsioon, proovi ilma selleta! See 5 on siin suvaline, seda peab vaatama veel. 
# fwhm = np.radians(7)
# delta_sm = hp.smoothing(delta, fwhm=fwhm, verbose=False)

# hp.mollview(m)
# hp.graticule()